In [1]:
# Run these small tutorials on CPU; select a GPU before importing JAX for larger runs.
import os
os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np
import marlax as mx
from marlax.envs import CooperativeGrid, GridState
from marlax.models import QTable
from marlax.methods import QLearning, SARSA, ExpectedSARSA


In [2]:
# 1. Define a one-step world with two actions per animal and one observation index.
# Both animals choosing action 1 earns a team reward. No grid-specific code is used.
from dataclasses import dataclass
@dataclass(frozen=True)
class Coordination:
    """A one-step cooperative game rewarding the joint action (1, 1)."""

    @property
    def spec(self):
        return mx.EnvSpec(2, 1, 2, "coordination_v1")

    def default_params(self):
        return jnp.array(1.0)

    def observe(self, state, params):
        return mx.PolicyInput(jnp.zeros(2, dtype=jnp.int32), jnp.ones((2, 2), dtype=bool))

    def global_state(self, state, params):
        return jnp.zeros(1)

    def reset(self, key, params):
        state = jnp.array(0)
        return state, mx.TimeStep(
            self.observe(state, params), self.global_state(state, params),
            jnp.zeros(2), jnp.array(False), jnp.array(False), {"coordinated": jnp.array(False)},
        )

    def step(self, key, state, actions, params):
        coordinated = jnp.all(actions == 1)
        state = state + 1
        return state, mx.TimeStep(
            self.observe(state, params), self.global_state(state, params),
            jnp.full(2, params * coordinated), jnp.array(True), jnp.array(False),
            {"coordinated": coordinated},
        )


In [3]:
# 2. Check the single-world interface before batching or learning.
env = Coordination()
params = env.default_params()
state, timestep = env.reset(jax.random.PRNGKey(0), params)
state, timestep = env.step(jax.random.PRNGKey(1), state, jnp.array([1, 1]), params)
assert timestep.terminated and timestep.metrics["coordinated"]
{"spec": env.spec, "reward": np.asarray(timestep.reward)}


{'spec': EnvSpec(num_agents=2, num_observations=1, num_actions=2, observation_schema='coordination_v1'),
 'reward': array([1., 1.], dtype=float32)}

In [4]:
# 3. Reuse the same QTable, agent, method, and runner as the grid experiments.
agent = mx.Agent(QTable(), sharing="shared")
method = ExpectedSARSA(alpha=0.2, epsilon_steps=4096)
run = mx.train(env, params, agent, method, seed=0, num_envs=32, env_steps=4096, chunk_steps=128)
report = mx.evaluate(run.policy, env, params, num_episodes=256, max_steps=1)
assert report["coordinated_rate"] == 1.0
report


{'cases': 256,
 'mean_return': 1.0,
 'mean_episode_steps': 1.0,
 'termination_rate': 1.0,
 'truncation_rate': 0.0,
 'unfinished_rate': 0.0,
 'coordinated_rate': 1.0}

In [5]:
# 4. Inspect the learned action preference; sharing needs only one parameter slot.
# Neural models require their own compatible learning method and are not implemented yet.
{"learned_q": np.asarray(run.policy.variables), "table_shape": run.policy.variables.shape}


{'learned_q': array([[[0.        , 0.96272254]]], dtype=float32),
 'table_shape': (1, 1, 2)}